AutoRAG Banner - Experiment Run

## Notebook content

This notebook presents the AutoRAG steps: data preparation, experiment execution, leaderboard analysis of the generated RAG patterns, and querying the selected pattern.

### Contents 
This notebook contains the following parts:
- **[Setup](#Setup)**
- **[Prepare experiment data](#Prepare-experiment-data)**
- **[Process input documents](#Process-input-documents)**
- **[Run ai4rag experiment](#Run-ai4rag-experiment)**
- **[Review experiment results](#review-experiment-results)**
- **[Summary](#Summary)**

## Setup

Install packages

In [ ]:
!pip install boto3 | tail -n 1
!pip install -U --no-cache-dir git+https://github.com/LukaszCmielowski/pipelines-components.git@rhoai_autorag | tail -n 1
!pip install docling | tail -n 1
!pip install "ai4rag==0.2.1" | tail -n 1

Import required libraries

In [ ]:
import re
import os
import json
import yaml
import logging
import urllib.request
from pathlib import Path
from types import SimpleNamespace

import warnings
warnings.filterwarnings("ignore")

import boto3
from langchain_core.documents import Document

for logger_name in (
        "Test Data Loader component logger",
        "Document Loader component logger",
        "Text Extraction component logger",
):
    logging.getLogger(logger_name).propagate = False

📌 **Action**: Provide the credentials for your S3 instance if they are not already set in the notebook environment.

💡 **Tip**: In the project, open Connections and add an S3 compatible object storage connection to a bucket you will use for documents and test data.
Open Workbenches, edit your workbench, and attach the S3 connection you created so the notebook can read from the bucket.
Save and restart the workbench if prompted.

In [ ]:
AWS_ACCESS_KEY_ID = ""
AWS_SECRET_ACCESS_KEY = ""
AWS_S3_ENDPOINT = ""
AWS_DEFAULT_REGION = ""

os.environ["AWS_ACCESS_KEY_ID"] = AWS_ACCESS_KEY_ID or os.environ.get("AWS_ACCESS_KEY_ID")
os.environ["AWS_SECRET_ACCESS_KEY"] = AWS_SECRET_ACCESS_KEY or os.environ.get("AWS_SECRET_ACCESS_KEY")
os.environ["AWS_S3_ENDPOINT"] = AWS_S3_ENDPOINT or os.environ.get("AWS_S3_ENDPOINT")
os.environ["AWS_DEFAULT_REGION"] = AWS_DEFAULT_REGION or os.environ.get("AWS_DEFAULT_REGION")

required_vars = ["AWS_ACCESS_KEY_ID", "AWS_SECRET_ACCESS_KEY", "AWS_S3_ENDPOINT", "AWS_DEFAULT_REGION"]
missing = [var for var in required_vars if not os.environ.get(var)]
if missing:
    raise ValueError(f"Missing required environment variables: {missing}")

📌 **Action**: Provide the bucket name where the experiment data will be stored.

> 🔖 **Note**: Bucket must already exist.

In [ ]:
BUCKET_NAME = ""

BUCKET_NAME = BUCKET_NAME or os.environ.get("BUCKET_NAME")
if not BUCKET_NAME:
    raise Exception("BUCKET_NAME must be provided")

## Prepare experiment data

### Initialize S3 client

In [ ]:
session = boto3.session.Session(
    aws_access_key_id=os.environ["AWS_ACCESS_KEY_ID"],
    aws_secret_access_key=os.environ["AWS_SECRET_ACCESS_KEY"],
)
s3_client = session.client(
    service_name='s3',
    endpoint_url=os.environ["AWS_S3_ENDPOINT"],
)

### Upload documents
For the needs of this notebook we are using IBM financial reports available here: https://www.ibm.com/investor/financial-reporting

In [ ]:
documents_urls = [
    "https://www.ibm.com/downloads/documents/us-en/12bb2f913a3ba1a2",
    "https://www.ibm.com/downloads/documents/us-en/131cf8a39db327fd",
    "https://www.ibm.com/downloads/documents/us-en/131cf87ab633199f",
    "https://www.ibm.com/downloads/documents/us-en/1550f7eea8c0ded6",
    "https://www.ibm.com/downloads/documents/us-en/10a9980400afd114",
    "https://www.ibm.com/downloads/documents/us-en/10a9980468afdf4c",
    "https://www.ibm.com/downloads/documents/us-en/10a9980400afd11c",
    "https://www.ibm.com/downloads/documents/us-en/11ed3283ae56ec71"
]

for url in documents_urls:
    with urllib.request.urlopen(url) as response:
        content = response.read()
        content_disposition = response.headers.get("Content-Disposition")
        filename = re.findall('filename="(.+)"', content_disposition)[0]
        s3_client.put_object(Bucket=BUCKET_NAME, Key=f"documents/{filename}", Body=content)

### Upload benchmark dataset
Benchmark data is required for proper evaluation of each created RAG pattern.
Every benchmark record needs to have the following fields:
- `"question"`: question that will be sent to the LLM and for which the retrieval will be made
- `"correct_answers"`: array (list in Python) of correct answers for the given question
- `"correct_answer_document_ids"`: filenames of the documents based on which the correct answers were created.

⚠️ **Warning**: Nested folders are currently not supported for the grounding documents in the selected S3 location.
Please use a single folder to store your documents.

⚠️ **Warning**: `document_id` corresponds to the document's filename.
That might change in the future.

In [ ]:
benchmark = [
    {
        "question": "What was IBM's revenue in the first quarter of 2024?",
        "correct_answers": [
            "Revenue of $14.5 billion, up 1 percent, up 3 percent at constant currency."
        ],
        "correct_answer_document_ids": [
            "ibm-1q25-earnings-press-release.pdf"
        ]
    },
    {
        "question": "What did IBM announce regarding HashiCorp in first quarter 2024?",
        "correct_answers": [
            "IBM announced its intent to acquire HashiCorp, Inc. for $35 per share in cash, representing an enterprise value of $6.4 billion. The transaction was expected to close by the end of 2024."
        ],
        "correct_answer_document_ids": [
            "ibm-1q25-earnings-press-release.pdf"
        ]
    },
        {
        "question": "How much did IBM invest in acquisitions in the first quarter of 2025, including HashiCorp?",
        "correct_answers": [
            "The company invested $7.1 billion in acquisitions, including the acquisition of HashiCorp."
        ],
        "correct_answer_document_ids": [
            "ibm-1q25-earnings-press-release.pdf"
        ]
    },
    {
        "question": "What quarterly dividend did the IBM board approve in July 2025?",
        "correct_answers": [
            "On July 23, 2025, the IBM board of directors approved a regular quarterly cash dividend of $1.68 per common share, to stockholders of record on August 8, 2025."
        ],
        "correct_answer_document_ids": [
            "ibm-2q25-earnings-press-release.pdf"
        ]
    },
    {
        "question": "What was IBM's third quarter 2024 revenue and Software growth?",
        "correct_answers": [
            "Revenue of $15.0 billion, up 1 percent, up 2 percent at constant currency. Software revenue up 10 percent."
        ],
        "correct_answer_document_ids": [
            "ibm-3q24-earnings-press-release.pdf"
        ]
    },
]

res = s3_client.put_object(Bucket=BUCKET_NAME, Key="benchmark.json", Body=json.dumps(benchmark))

Look up bucket contents

In [ ]:
res = s3_client.list_objects_v2(
    Bucket=BUCKET_NAME,
    Prefix="",
).get("Contents", [])

print("Bucket contents:")
for r in res:
    print(r["Key"])

## Process input documents

The data processing flow prepares input for the experiment in three steps. Each step runs as a standalone component (via `python_func`) with artifact paths under `step_outputs/`. Ensure `AWS_ACCESS_KEY_ID`, `AWS_SECRET_ACCESS_KEY`, `AWS_S3_ENDPOINT`, and `AWS_DEFAULT_REGION` are set in your environment before running.

| Step | Component | Purpose |
|------|-----------|---------|
| 1 | **Test data loader** | Download the benchmark JSON (questions and ground truth) from S3. |
| 2 | **Documents sampling** | List documents in the bucket, prioritize benchmark-referenced docs, apply a size cap, and write a YAML manifest (no content download). |
| 3 | **Text extraction** | Download the listed documents from S3 and extract text to Markdown using Docling. |

In [ ]:
from kfp_components.components.data_processing.autorag.test_data_loader.component import test_data_loader
from kfp_components.components.data_processing.autorag.documents_sampling.component import documents_sampling
from kfp_components.components.data_processing.autorag.text_extraction.component import text_extraction

step_output_dir = Path("./step_outputs")
step_output_dir.mkdir(parents=True, exist_ok=True)

test_data_bucket_name = BUCKET_NAME
test_data_key = "benchmark.json"
input_data_bucket_name = BUCKET_NAME
input_data_key = ""
sampling_config = {}

#### Step 1: Test data loader

Downloads the benchmark (test data) JSON file from the configured S3 bucket and key. The file must be a JSON containing a list of items with `question`, `correct_answers`, and `correct_answer_document_ids`.

In [ ]:
test_data_out = SimpleNamespace(path=str(step_output_dir / "test_data.json"))

test_data_loader.python_func(
    test_data_bucket_name=test_data_bucket_name,
    test_data_path=test_data_key,
    test_data=test_data_out,
)

output_path = Path(test_data_out.path)
with output_path.open("r", encoding="utf-8") as f:
    test_data = json.load(f)

print(json.dumps(test_data, indent=4, ensure_ascii=False))

#### Step 2: Documents sampling

Lists objects in the S3 input bucket, filters by supported extensions (e.g. `.pdf`, `.docx`, `.pptx`, `.md`, `.html`, `.txt`), and builds a sampled set: documents referenced in the benchmark (from Step 1) are prioritized; then others are added until a configurable size limit (1 GB by default) is reached. **This step does not download document contents.** It writes a YAML manifest, `sampled_documents_descriptor.yaml`, containing bucket, prefix, and the list of selected object keys and sizes. That manifest is the input for the text extraction step.

In [ ]:
test_data_in = SimpleNamespace(path=str(step_output_dir / "test_data.json"))
sampled_documents_out = SimpleNamespace(path=str(step_output_dir / "sampled_documents"))

documents_sampling.python_func(
    input_data_bucket_name=input_data_bucket_name,
    input_data_path=input_data_key,
    test_data=test_data_in,
    sampling_config=sampling_config,
    sampled_documents=sampled_documents_out,
)

descriptor_path = step_output_dir / "sampled_documents" / "sampled_documents_descriptor.yaml"
with open(descriptor_path) as f:
    descriptor = yaml.safe_load(f)

print(json.dumps(descriptor, indent=4, ensure_ascii=False))

#### Step 3: Text extraction

Reads the `sampled_documents_descriptor.yaml` produced by Step 2, downloads each listed document from S3 into a temporary directory, and runs **Docling** to extract text. Output is one Markdown file per document (e.g. `document_0.md`, `document_1.md`) written to the artifact output path. These files are the final text corpus for the experiment.

In [ ]:
sampled_descriptor_in = SimpleNamespace(path=str(step_output_dir / "sampled_documents"))
extracted_text_out = SimpleNamespace(path=str(step_output_dir / "extracted_text"))

text_extraction.python_func(
    sampled_documents_descriptor=sampled_descriptor_in,
    extracted_text=extracted_text_out,
)

Load the extracted Markdown files from Step 3 into LangChain `Document` objects.
> 🔖 **Note:** Document metadata must contain the `document_id` key with the name of the document as it was referenced in the benchmark data JSON file.

In [ ]:
paths = list(Path("step_outputs/extracted_text").glob("*.md"))
documents = [
    Document(
        page_content=p.read_text(encoding="utf-8", errors="replace"),
        metadata={"document_id": p.stem},
    )
    for p in sorted(paths)
]

n = 3
print(f"First {n} documents:")
for doc in documents[:n]:
    print("=" * 100)
    print(doc.metadata)
    print(doc.page_content[:800])


## Run ai4rag experiment

### Prepare LlamaStackClient

The following sections allow you to run the `ai4rag` experiment using Llama Stack.
To do so, you need to provide an instance of the `LlamaStackClient`.
To instantiate the client, you must provide `LLAMA_STACK_CLIENT_API_KEY` and `LLAMA_STACK_CLIENT_BASE_URL` environment variables or provide them in the password prompt after running the cell below.

🔖 **Note**: To use `ai4rag` with Llama Stack, at least 1 foundation model, 1 embedding model, and a connected Milvus instance should be available on the server side.

In [ ]:
import os
import getpass

from llama_stack_client import LlamaStackClient

if not os.getenv("LLAMA_STACK_CLIENT_API_KEY") or not os.getenv("LLAMA_STACK_CLIENT_BASE_URL"):
    os.environ["LLAMA_STACK_CLIENT_API_KEY"] = getpass.getpass("Please enter 'LLAMA_STACK_CLIENT_API_KEY': ")
    os.environ["LLAMA_STACK_CLIENT_BASE_URL"] = getpass.getpass("Please enter 'LLAMA_STACK_CLIENT_BASE_URL': ")

client = LlamaStackClient(
    base_url=os.getenv("LLAMA_STACK_CLIENT_BASE_URL"),
    api_key=os.getenv("LLAMA_STACK_CLIENT_API_KEY"),
)

You can list the available models using `client.models.list()`.

In [ ]:
client.models.list()

### Configure the search space

📌 **Action**: You need to select foundation models and embedding models. You can use the models listed above. A foundation model requires only `model_id` and `client`. An embedding model requires you to provide `params` as a `dict` with at least 1 parameter: `embedding_dimension`. This information can be extracted from the embedding model metadata.

In [ ]:
from ai4rag.rag.embedding.llama_stack import LSEmbeddingModel
from ai4rag.rag.foundation_models.llama_stack import LSFoundationModel
from ai4rag.search_space.src.parameter import Parameter

foundation_models = [LSFoundationModel(model_id="vllm-inference-llama-3-1/redhataillama-31-8b-instruct", client=client)]

embedding_models = [
    LSEmbeddingModel(
        model_id="vllm-embedding/granite-278m-multilingual-1",
        client=client,
        params={"embedding_dimension": 768},
    ),
]

Now you can create a search space that will be used during the experiment.

In [ ]:
from ai4rag.search_space.src.search_space import AI4RAGSearchSpace

search_space = AI4RAGSearchSpace(
    params=[
        Parameter(name="foundation_model", param_type="C", values=foundation_models),
        Parameter(name="embedding_model", param_type="C", values=embedding_models),
    ]
)

print(f"Maximum possible combinations: {search_space.max_combinations}")

### Configure optimizer

The experiment uses `GAMOptimizer`.
It can be configured with `GAMOptSettings` by changing 2 values:
- `max_evals` describes the maximum number of evaluations that the algorithm will run (1 evaluation = 1 RAG pattern built and evaluated)
- `n_random_nodes` describes the number of random nodes in the search space to evaluate first. The greater the value, the better the chance to avoid local minima and improve search space exploration.

In [ ]:
from ai4rag.core.hpo.gam_opt import GAMOptSettings

optimizer_settings = GAMOptSettings(max_evals=10, n_random_nodes=4)

### Create and run the experiment

Now you are ready to create and run your experiment. There are several parameters in the `AI4RAGExperiment` class:
- `client`: expects an instance of `LlamaStackClient` that will be used for communication with the Milvus vector database
- `documents`: documents with proper metadata used as a knowledge base for the LLM
- `benchmark_data`: records to properly evaluate the LLM's responses
- `search_space`: defined parameters that create the space of RAG patterns
- `optimizer_settings`: controls the optimization process
- `optimization_metric`: defines the metric for which the RAG pattern will be optimized
- `event_handler`: allows you to monitor status updates and handle results after each iteration
- `vector_store_type`: defines which vector database to use (`"ls_milvus"` uses Llama Stack Milvus)

In [ ]:
import pandas as pd

from ai4rag.core.experiment.experiment import AI4RAGExperiment
from ai4rag.utils.event_handler import LocalEventHandler

experiment = AI4RAGExperiment(
    client=client,
    documents=documents,
    benchmark_data=pd.DataFrame(test_data),
    search_space=search_space,
    optimizer_settings=optimizer_settings,
    optimization_metric="faithfulness",
    event_handler=LocalEventHandler(output_path=Path(".").absolute() / "ai4rag_results"),
    vector_store_type="ls_milvus",
)

experiment.search()

## Review experiment results

Display the evaluation results in a clean, tabular format showing scores, parameters, and metadata.

In [ ]:
from IPython.display import display

best_evals = experiment.results.get_best_evaluations(k=3)

results_df = pd.DataFrame(
    [
        {
            "Pattern Name": b_eval.pattern_name,
            **{k: v["mean"] for k, v in b_eval.scores["scores"].items()},
            "Foundation Model": b_eval.rag_params["generation"]["model_id"],
            "Embedding Model": b_eval.indexing_params["embedding"]["model_id"],
            "Chunking Settings": b_eval.indexing_params["chunking"], "Retrieval Settings": b_eval.rag_params["retrieval"]
        } for
     b_eval in best_evals],
)

display(results_df)



Query the RAG pattern and display the answer along with grounding documents.

In [ ]:
def query_rag_pattern(rag_pattern, **retrieval_kwargs):
    """
    Query the RAG pattern and display the answer with grounding documents.
    
    Parameters
    ----------
    rag_pattern : LlamaStackRAG
        RAG pattern that is able to retrieve proper context
        and respond to the question.

    Returns
    -------
    dict
    Response containing answer, reference_documents, and question
    """
    question = input("Question: ")
    print(f"\n{'=' * 80}")
    print(f"Question: {question}")
    print(f"{'=' * 80}\n")

    response = rag_pattern.generate(question, **retrieval_kwargs)

    print("Answer:")
    print(f"{response['answer']}\n")

    print(f"{'=' * 80}")
    print(f"Grounding Documents ({len(response['reference_documents'])} retrieved):")
    print(f"{'=' * 80}\n")

    for idx, doc in enumerate(response['reference_documents'], 1):
        doc_id = doc.metadata.get('document_id', 'Unknown')
        content_preview = doc.page_content[:300].replace('\n', ' ')
        if len(doc.page_content) > 300:
            content_preview += "..."

        print(f"[{idx}] Document ID: {doc_id}")
        print(f"    Content: {content_preview}")
        print()

    return response

In [ ]:
query_rag_pattern(rag_pattern=best_evals[0].rag_pattern)

## Summary

**Summary:** This notebook set up the experiment data, processed it, ran the `ai4rag` experiment, displayed the RAG patterns leaderboard, and queried the selected pattern.

**Authors**
- **Jakub Walaszczyk**, Software Engineer @ Red Hat
- **Witold Nowogórski**, Associate Software Engineer @ Red Hat
